# State_management

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph , START, END

class badstate(TypedDict):
    steps : list


def first_step(state : badstate) -> dict:
    current  =  state['steps']
    new_step = current + ['step_one']

    print(f'curent step is {new_step}')

    return{ 'steps':new_step}

In [2]:
def second_step(state:badstate) -> dict:
    current  =  state['steps']
    new_step = current + ['step_two']
    print(f'curent step is {new_step}')

    return{ 'steps':new_step}
    

In [3]:

builder = StateGraph(badstate)

builder.add_node('first_step',first_step)
builder.add_node('second_step',second_step)

builder.add_edge(START,'first_step')
builder.add_edge('first_step','second_step')
builder.add_edge('second_step',END)

In [4]:
graph = builder.compile()
result = graph.invoke({"steps": []})
print("Final step:", result["steps"])

curent step is ['step_one']
curent step is ['step_one', 'step_two']
Final step: ['step_one', 'step_two']


In [6]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph , START, END
from operator import add

class GoodListState(TypedDict):
    steps : Annotated[list,add]



In [8]:
def first_step(state: GoodListState) -> dict:
    print(f"  step_one sees steps={state['steps']}")
    # Return ONLY the new item(s) - NOT the whole list
    return {"steps": ["first_step"]}


def second_step(state: GoodListState) -> dict:
    print(f"  step_two sees steps={state['steps']}")
    # Return ONLY the new item(s)
    return {"steps": ["second_step"]}


In [9]:

builder = StateGraph(GoodListState)

builder.add_node('first_step',first_step)
builder.add_node('second_step',second_step)

builder.add_edge(START,'first_step')
builder.add_edge('first_step','second_step')
builder.add_edge('second_step',END)

In [10]:
graph = builder.compile()
result = graph.invoke({"steps": []})
print("Final step:", result["steps"])

  step_one sees steps=[]
  step_two sees steps=['first_step']
Final step: ['first_step', 'second_step']


In [12]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class chatstate(TypedDict):
    message : Annotated[list , add_messages]
    turn : int


In [14]:
def human_ques(state : chatstate) -> dict:
    new_mess = HumanMessage(content='what is capital of india')
    print(f'[human_input],appending{new_mess.content}')

    return{
        'message':new_mess,
        'turn':state['turn']+1
        
    }


def ai_reply(state : chatstate) -> dict:
    new_mess =  AIMessage(content='the capital of india is newdelhi')
    print(f'[ai_reply],appending{new_mess.content}')

    return{
        'message':new_mess
    }


